In [85]:
# ==============================================================================
# 라이브러리 및 모듈 임포트
# ==============================================================================
import time  # 페이지 로딩 대기 및 서버 부하 방지를 위한 시간 지연
import json  # 수집된 데이터를 JSON 형식으로 변환 및 저장
import os    # 파일 경로 생성 및 디렉토리 확인을 위한 시스템 모듈
from selenium import webdriver  # 동적 웹페이지 제어를 위한 자동화 도구
from selenium.webdriver.chrome.options import Options  # 브라우저 환경 설정을 위한 옵션
from bs4 import BeautifulSoup  # HTML 데이터를 파싱하여 원하는 태그 추출
from concurrent.futures import ThreadPoolExecutor  # 병렬 처리를 통해 수집 속도 최적화

In [66]:
# 브라우저 환경 최적화 함수
def get_safe_driver():
    # Chrome 브라우저 실행 옵션 설정
    chrome_options = Options()
    # 서버 환경이나 리소스 절약을 위해 샌드박스 및 GPU 비활성화
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    # 이미지 로딩을 차단하여 페이지 로드 속도 획기적으로 개선
    chrome_options.add_argument("--blink-settings=imagesEnabled=false")
    # 페이지 로드 전략을 eager(DOM 완성 시점)로 설정하여 불필요한 대기 시간 제거
    chrome_options.page_load_strategy = 'eager'
    # 봇 차단을 방지하기 위한 사용자 에이전트 설정
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [90]:
class RecipeCrawler:
    def __init__(self, save_path="./recipes3.js"):
        # 데이터가 저장될 위치 초기화
        self.save_path = save_path
        # 수집 대상 카테고리 목록 정의
        self.categories = ["한식", "중식", "일식", "양식", "분식", "디저트"]

    # 개별 레시피 상세 페이지를 수집하는 작업 단위 함수
    def fetch_detail_worker(self, title, url, thumbnail, category):
        worker_driver = None
        try:
            # 병렬 처리를 위해 각 스레드마다 독립적인 브라우저 드라이버 생성
            worker_driver = get_safe_driver()
            worker_driver.get(url)
            time.sleep(2) # 상세 페이지 로딩 대기
            soup = BeautifulSoup(worker_driver.page_source, 'html.parser')
            
            # 재료 가공: 리스트 내 데이터 추출 및 불필요한 텍스트 제거
            ingredients_tags = soup.select("#divConfirmedMaterialArea > ul:nth-child(1) > li")
            ingredients = []
            for i, tag in enumerate(ingredients_tags, 1):
                raw_text = " ".join(tag.get_text(separator=' ', strip=True).split())
                raw_text = raw_text.replace("구매", "").strip()
                if raw_text and len(raw_text) > 1:
                    ingredients.append({"ingredientId": i, "amount": raw_text})
            
            # 조리 순서 가공: 이미지 포함 레시피 단계 추출
            step_divs = soup.select('#obx_recipe_step_start .view_step_cont')
            steps = []
            for i, div in enumerate(step_divs, 1):
                desc = div.get_text(separator=' ', strip=True)
                if "관련 상품" in desc or "등록일" in desc or not desc: continue
                img_tag = div.select_one("img")
                steps.append({"stepNo": i, "description": desc, "cookingImageUrl": img_tag['src'] if img_tag else ""})

            # 가공된 데이터 객체 반환
            return {
                "title": title,
                "category": category, # 프론트엔드 필터링을 위한 카테고리 필드 추가
                "description": f"{category} 스타일의 {title} 레시피입니다.",
                "thumbnailImageUrl": thumbnail,
                "ingredients": ingredients,
                "steps": steps
            }
        except Exception as e:
            print(f"❌ 에러 발생 [{title}]: {e}")
            return None
        finally:
            if worker_driver: worker_driver.quit() # 자원 해제
        
    def run(self):
        all_results = []
        
        # [흐름 1] 카테고리별 검색 결과 페이지 분석 루프
        for category in self.categories:
            print(f"🚀 [Step 1] '{category}' 카테고리 검색 페이지 분석 중...")
            main_driver = get_safe_driver()
            try:
                # 검색 페이지 URL 구성
                search_url = f"https://www.10000recipe.com/recipe/list.html?q={category}"
                main_driver.get(search_url)
                time.sleep(3)
                soup = BeautifulSoup(main_driver.page_source, 'html.parser')
                
                # 리스트 아이템 추출 (카테고리당 3개 수집 예시)
                items = soup.select("ul.common_sp_list_ul.ea4 > li")[:3]
                
                for item in items:
                    title_tag = item.select_one(".common_sp_caption_tit")
                    link_tag = item.select_one(".common_sp_thumb > a")
                    img_tag = item.select_one(".common_sp_thumb > a > img")
                    
                    if title_tag and link_tag:
                        # 병렬 작업을 위한 튜플 데이터 저장
                        all_results.append((
                            title_tag.get_text(strip=True),
                            "https://www.10000recipe.com" + link_tag['href'],
                            img_tag['src'] if img_tag else "",
                            category
                        ))
            finally:
                main_driver.quit()

        # [흐름 2] 상세 페이지 병렬 수집 (ThreadPoolExecutor로 효율성 증대)
        print(f"🚀 [Step 2] {len(all_results)}개 레시피 상세 수집 시작...")
        with ThreadPoolExecutor(max_workers=2) as executor:
            results = list(executor.map(lambda x: self.fetch_detail_worker(x[0], x[1], x[2], x[3]), all_results))
            final_results = [r for r in results if r]

        # [흐름 3] 데이터를 JS 모듈 파일로 저장
        os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
        js_content = f"export const recipes3 = {json.dumps(final_results, indent=2, ensure_ascii=False)};"
        with open(self.save_path, "w", encoding="utf-8") as f:
            f.write(js_content)
        
        print(f"\n✨ [완료] {len(final_results)}개의 레시피 데이터가 {self.save_path}에 저장되었습니다!")

In [91]:
if __name__ == "__main__":
    crawler = RecipeCrawler()
    crawler.run()

🚀 [Step 1] '한식' 카테고리 검색 페이지 분석 중...
🚀 [Step 1] '중식' 카테고리 검색 페이지 분석 중...
🚀 [Step 1] '일식' 카테고리 검색 페이지 분석 중...
🚀 [Step 1] '양식' 카테고리 검색 페이지 분석 중...
🚀 [Step 1] '분식' 카테고리 검색 페이지 분석 중...
🚀 [Step 1] '디저트' 카테고리 검색 페이지 분석 중...
🚀 [Step 2] 18개 레시피 상세 수집 시작...

✨ [완료] 18개의 레시피 데이터가 ./recipes3.js에 저장되었습니다!
